# Proyecto final NLP · Prototipo
## Propuesta asistida de familia de material para la declaración de envases (Punto Verde)

**Caso:** cada componente de envase nuevo llega al BOM con una descripción libre (`FILM BOPP MET 20MIC 5,2G`, `MEMBRANA PET/ALU/PE`, `TAMPA PEAD 28MM`...). El analista de sostenibilidad debe decidir en qué **familia de material** se declara y con qué **peso**. Hoy se hace leyendo cada descripción a mano.

**Decisión que se apoya:** qué familia y peso se proponen para revisión antes de cargar la declaración. El sistema **no declara**: propone, justifica con casos similares y marca lo que debe revisar una persona.

**Datos:** conjunto **ficticio** de 189 descripciones generado para la demostración (`componentes_envase_ficticio.csv`). No contiene datos internos.

| Parte | Capacidad | Pregunta | Reutiliza |
|---|---|---|---|
| A | Extraer | ¿Qué peso declara la descripción? | Notebook 04 (NER general vs dominio) |
| B | Clasificar | ¿Qué familia de material es? | Notebooks 01 (reglas) y 02 (TF-IDF + clasificador) |
| C | Encontrar | ¿Qué componentes ya validados se parecen? | Notebook 03 (búsqueda léxica vs semántica) |
| D | Integrar | ¿Qué ve el analista? | Combinación de A + B + C |

In [1]:
# Ejecuta esta celda una vez. En Colab instala lo necesario; en local usa el entorno del curso.
import subprocess, sys
try:
    import google.colab  # type: ignore
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
if EN_COLAB:
    paquetes = [
        "sentence-transformers==6.0.1", "spacy==3.8.16",
        "https://github.com/explosion/spacy-models/releases/download/es_core_news_sm-3.8.0/es_core_news_sm-3.8.0-py3-none-any.whl",
    ]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *paquetes], check=True)
    print("✅ Colab preparado. Sube 'componentes_envase_ficticio.csv' al panel de archivos.")
else:
    print("✅ Entorno local")

✅ Entorno local


## 0. Datos de entrada

In [2]:
import re
import numpy as np
import pandas as pd

RUTA = "componentes_envase_ficticio.csv"
df = pd.read_csv(RUTA, sep=";")
print(df.shape)
df.head(8)

(189, 8)


,id_componente,descripcion,familia,peso_g,proveedor,idioma,caso_dificil,motivo_dificultad
0,CMP-1000,BOT. PET TRANSP. 1.25L 38.2G,PET,38.2,PROV_B,ES,False,NaN
1,CMP-1001,BRICK CARTON/PE/ALU 1L 28G,COMPLEJO_ALU,28.0,PROV_D,ES,True,Contiene 'CARTON' pero es envase complejo
2,CMP-1002,BOLSA PE BAJA DENSIDAD 20µ,PEBD,56.3,PROV_A,ES,False,NaN
3,CMP-1003,"TARRINA PP 200G 0,0108KG",PP,10.8,PROV_B,ES,False,NaN
4,CMP-1004,PREFORMA PET 37.7G CUELLO PCO1881,PET,37.7,PROV_E,ES,False,NaN
5,CMP-1005,BOLSA PE RECICLADO 30% PCR 60MIC 18G,PEBD,18.0,PROV_D,ES,True,Porcentaje de reciclado puede confundirse con ...
6,CMP-1006,FILM ESTIRABLE PE 20 MIC PALET,PEBD,48.1,PROV_B,ES,False,NaN
7,CMP-1007,"TAPON CORONA ALUMINIO 26MM 1,9G",ALUMINIO,1.9,PROV_F,ES,True,"Tapón corona normalmente acero, aquí aluminio"


In [3]:
resumen = pd.concat(
    [df["familia"].value_counts().rename("componentes"),
     df.groupby("familia")["caso_dificil"].sum().rename("casos difíciles")], axis=1)
print("Idiomas:", df["idioma"].value_counts().to_dict())
resumen

Idiomas: {'ES': 142, 'EN': 37, 'PT': 10}


,componentes,casos difíciles
familia,,
PAPEL_CARTON,31,5
PET,27,2
PP,22,3
PLASTICO_MULTICAPA,22,3
PEBD,20,2
ALUMINIO,19,2
COMPLEJO_ALU,16,3
PEAD,14,0
ACERO,9,1


### Guía de anotación (criterios internos de la demostración)

Las etiquetas no son la tabla oficial de Ecoembes: son una **taxonomía simplificada**. Igual que en el notebook 04 (sentimiento), una etiqueta solo es evaluable si antes definimos qué significa.

| Situación | Criterio adoptado |
|---|---|
| Laminado con aluminio (PET/ALU, PAPEL/ALU, brick) | `COMPLEJO_ALU` |
| Laminado solo plástico o plástico metalizado (BOPP MET) | `PLASTICO_MULTICAPA` |
| Componente con un elemento minoritario de otro material (tapón PP con liner ALU) | Material mayoritario |
| Dos componentes separables (bandeja cartón + film) | Material mayoritario en peso |
| Papel metalizado | `PAPEL_CARTON` |

# Parte A — Extraer el peso

**Predicción:** en `LAMINADO OPP/MET-OPP 150G PATATA 8,5G`, ¿qué número es el peso del envase? Una persona sabe que `150G` es el contenido del producto. ¿Lo sabrá un modelo general?

Primero probamos el NER general de spaCy (notebook 04).

In [4]:
import spacy
nlp = spacy.load("es_core_news_sm")
ejemplos_ner = [
    "LAMINADO OPP/MET-OPP 150G PATATA 8,5G",
    "TAPON PEAD 28MM 2,4G",
    "CAJA CARTON 0,215KG",
]
pd.DataFrame([
    {"Descripción": t, "Entidades spaCy": [(e.text, e.label_) for e in nlp(t).ents]}
    for t in ejemplos_ner
])

,Descripción,Entidades spaCy
0,"LAMINADO OPP/MET-OPP 150G PATATA 8,5G","[(OPP, ORG)]"
1,"TAPON PEAD 28MM 2,4G","[(PEAD, MISC)]"
2,"CAJA CARTON 0,215KG","[(CAJA CARTON, LOC)]"


El NER general solo conoce `PER`, `ORG`, `LOC` y `MISC`. No tiene la categoría *peso del envase* y, al ver texto en mayúsculas, etiqueta fragmentos sin sentido para el dominio (un material como organización, una caja como lugar). No es un fallo del modelo: se entrenó con noticias, no con descripciones de BOM. Como el formato del peso es **relativamente estable** (número + unidad), probamos una **expresión regular**: es la técnica suficiente para un patrón de dominio, como concluía el notebook 04.

Comparamos dos versiones:

- **Ingenua:** primer número seguido de `G`.
- **Refinada:** solo `G`, `GR` o `KG` como palabra completa (descarta `MIC`, `MM`, `CL`), ignora `12X150G` (unidades por caja), convierte kg a g y, si hay varias menciones, toma la **última** y activa un **aviso**.

In [5]:
PATRON_PESO = re.compile(r"(?<![\dX])(\d+(?:[.,]\d+)?)\s?(KG|GR|G)\b", re.IGNORECASE)

def peso_ingenuo(texto):
    m = re.search(r"(\d+(?:[.,]\d+)?)\s?G", texto, re.IGNORECASE)
    return float(m.group(1).replace(",", ".")) if m else None

def extraer_peso(texto):
    menciones = [float(n.replace(",", ".")) * (1000 if u.upper() == "KG" else 1)
                 for n, u in PATRON_PESO.findall(texto)]
    if not menciones:
        return None, "sin peso en la descripción"
    aviso = "varias menciones: revisar" if len(menciones) > 1 else ""
    return round(menciones[-1], 1), aviso

for t in ["LAMINADO OPP/MET-OPP 150G PATATA 8,5G", "FILM 23MIC TRANSP. PALETIZADO",
          "CAJA CARTON 0,215KG", "CAJA CARTON ONDULADO 12X150G 240G"]:
    print(f"{t!r:45} ingenuo={peso_ingenuo(t)}  refinado={extraer_peso(t)}")

'LAMINADO OPP/MET-OPP 150G PATATA 8,5G'       ingenuo=150.0  refinado=(8.5, 'varias menciones: revisar')
'FILM 23MIC TRANSP. PALETIZADO'               ingenuo=None  refinado=(None, 'sin peso en la descripción')
'CAJA CARTON 0,215KG'                         ingenuo=None  refinado=(215.0, '')
'CAJA CARTON ONDULADO 12X150G 240G'           ingenuo=150.0  refinado=(240.0, '')


In [6]:
def correcto(pred, real):
    return pred is not None and not pd.isna(pred) and abs(pred - real) < 0.05

df["peso_ingenuo"] = df["descripcion"].map(peso_ingenuo)
df[["peso_extraido", "aviso_peso"]] = df["descripcion"].apply(lambda t: pd.Series(extraer_peso(t)))

def resumen_extraccion(col):
    extrae = df[col].notna()
    acierta = df.apply(lambda r: correcto(r[col], r["peso_g"]), axis=1)
    return {"Extrae un valor": extrae.mean(), "Valor correcto": acierta.mean(),
            "Valor INCORRECTO (error silencioso)": (extrae & ~acierta).mean()}

pd.DataFrame({"Regex ingenua": resumen_extraccion("peso_ingenuo"),
              "Regex refinada": resumen_extraccion("peso_extraido")}).style.format("{:.1%}")

,Regex ingenua,Regex refinada
Extrae un valor,68.3%,78.8%
Valor correcto,56.6%,77.2%
Valor INCORRECTO (error silencioso),11.6%,1.6%


In [7]:
errores_peso = df[df["peso_extraido"].notna() &
                  ~df.apply(lambda r: correcto(r["peso_extraido"], r["peso_g"]), axis=1)]
print("Errores silenciosos de la regex refinada:")
errores_peso[["descripcion", "peso_g", "peso_extraido", "aviso_peso"]]

Errores silenciosos de la regex refinada:


,descripcion,peso_g,peso_extraido,aviso_peso
24,LATA HOJALATA 500G,55.0,500.0,
92,LAMINADO OPP/MET-OPP 150G PATATA,9.4,150.0,
100,ESTUCHE CARTONCILLO CEREAL 375G,58.2,375.0,


**Interpretación.** La regex refinada no inventa pesos donde no los hay (espesores, diámetros, volúmenes) y gestiona g/kg y comas decimales. Los errores que quedan comparten causa: la descripción **solo** contiene el peso del contenido (`500G`, `375G`), no el del envase. Ninguna regla puede distinguirlo sin contexto adicional, y el aviso no salta porque solo hay una mención.

**Limitación y control.** El error más caro es un peso plausible pero falso, porque pasa desapercibido. Control propuesto: contrastar el peso extraído con el rango habitual de la familia o con la ficha técnica del proveedor, y priorizar siempre la columna estructurada de peso del BOM cuando exista.

# Parte B — Clasificar la familia de material

## B1. Baselines: clase mayoritaria y reglas

Primero, la referencia ingenua: asignar siempre la familia más frecuente. Después, un clasificador de **reglas** al estilo del notebook 01, como lo escribiría un analista: palabras clave por material con un orden de prioridad.

**Predicción:** ¿qué hará la regla con `MEMBRANA SIN ALU PE/PET` o con `BOLSA ALU-FREE OPP/PE`?

In [8]:
def clasificar_regla(texto):
    t = texto.upper()
    if "ALU" in t and ("/" in t or "+" in t):
        return "COMPLEJO_ALU"
    if "ALU" in t:
        return "ALUMINIO"
    if "ACERO" in t or "HOJALATA" in t:
        return "ACERO"
    if "VIDRIO" in t or "GLASS" in t:
        return "VIDRIO"
    if "/" in t or "MET" in t or "LAMINADO" in t:
        return "PLASTICO_MULTICAPA"
    if "PEAD" in t or "HDPE" in t:
        return "PEAD"
    if "PEBD" in t or "LDPE" in t:
        return "PEBD"
    if "PET" in t:
        return "PET"
    if "PP" in t:
        return "PP"
    if any(k in t for k in ["CARTON", "CARTAO", "CORRUGATED", "PAPEL"]):
        return "PAPEL_CARTON"
    return "SIN_REGLA"

for t in ["LATA ALU 33CL 11,2G", "MEMBRANA SIN ALU PE/PET 1,1G", "BOLSA ALU-FREE OPP/PE 6,4G", "BOT. VIDR. 33CL 0,21KG"]:
    print(f"{t!r:35} → {clasificar_regla(t)}")

'LATA ALU 33CL 11,2G'               → ALUMINIO
'MEMBRANA SIN ALU PE/PET 1,1G'      → COMPLEJO_ALU
'BOLSA ALU-FREE OPP/PE 6,4G'        → COMPLEJO_ALU
'BOT. VIDR. 33CL 0,21KG'            → SIN_REGLA


## B2. Preparación: ¿qué conservar?

En el notebook 02 vimos que eliminar *stopwords* puede borrar `no`. Aquí ocurre lo mismo con `sin`:

In [9]:
doc = nlp("MEMBRANA SIN ALU PE/PET")
[(t.text, t.is_stop) for t in doc]

[('MEMBRANA', False),
 ('SIN', True),
 ('ALU', False),
 ('PE', False),
 ('/', False),
 ('PET', False)]

Si eliminamos stopwords, `MEMBRANA SIN ALU` y `MEMBRANA ALU` quedarían casi iguales y cambiaría la familia. **Decisión:** no eliminamos stopwords ni lematizamos (son códigos, no frases), pero sí separamos por `/` y `+`, porque en estas descripciones indican capas de material.

## B3. TF-IDF + regresión logística (notebook 02, parte 4)

Comparamos dos representaciones con el mismo clasificador:

- **TF-IDF de palabras** (unigramas y bigramas): `PET`, `ALU`, `SIN ALU`...
- **TF-IDF de caracteres** (2–5 caracteres): robusto a erratas (`ALUMNIO`), abreviaturas (`VIDR.`) y mezcla de idiomas (`CARTON`/`CARTAO`).

Con 189 ejemplos una sola partición sería inestable, así que usamos **validación cruzada de 5 particiones**: cada descripción se predice con un modelo que no la vio al entrenar.

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score, recall_score

def vec_palabras():
    return TfidfVectorizer(ngram_range=(1, 2), token_pattern=r"[^\s/+]+")

def vec_caracteres():
    return TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5))

def clasificador(vec):
    return make_pipeline(vec, LogisticRegression(max_iter=3000, C=10, class_weight="balanced"))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
CLASES = np.array(sorted(df["familia"].unique()))

proba_pal = cross_val_predict(clasificador(vec_palabras()), df["descripcion"], df["familia"], cv=cv, method="predict_proba")
proba_car = cross_val_predict(clasificador(vec_caracteres()), df["descripcion"], df["familia"], cv=cv, method="predict_proba")

df["pred_mayoritaria"] = df["familia"].mode()[0]
df["pred_regla"] = df["descripcion"].map(clasificar_regla)
df["pred_tfidf_palabras"] = CLASES[proba_pal.argmax(1)]
df["pred_tfidf_caracteres"] = CLASES[proba_car.argmax(1)]
df["score_caracteres"] = proba_car.max(1)

In [11]:
dificil = df["caso_dificil"]
filas = []
for nombre, col in [("Clase mayoritaria", "pred_mayoritaria"), ("Reglas", "pred_regla"),
                    ("TF-IDF palabras + LR", "pred_tfidf_palabras"), ("TF-IDF caracteres + LR", "pred_tfidf_caracteres")]:
    filas.append({
        "Método": nombre,
        "Accuracy": accuracy_score(df["familia"], df[col]),
        "F1 macro": f1_score(df["familia"], df[col], average="macro"),
        "Accuracy casos difíciles (n=22)": accuracy_score(df.loc[dificil, "familia"], df.loc[dificil, col]),
        "Recall COMPLEJO_ALU": recall_score(df["familia"], df[col], labels=["COMPLEJO_ALU"], average="macro"),
    })
comparativa = pd.DataFrame(filas).set_index("Método")
comparativa.style.format("{:.1%}")

,Accuracy,F1 macro,Accuracy casos difíciles (n=22),Recall COMPLEJO_ALU
Método,,,,
Clase mayoritaria,16.4%,2.8%,22.7%,0.0%
Reglas,88.4%,83.8%,40.9%,100.0%
TF-IDF palabras + LR,92.1%,91.2%,50.0%,93.8%
TF-IDF caracteres + LR,92.6%,92.3%,59.1%,81.2%


**Cómo leer la tabla.** La *accuracy* global se parece entre reglas y modelos porque la mayoría de descripciones son regulares. La diferencia aparece en **F1 macro** (trata igual a las familias pequeñas) y en los **casos difíciles**. El recall de `COMPLEJO_ALU` importa porque declarar un complejo como monomaterial es el error más caro del caso.

Veamos los casos difíciles uno a uno:

In [12]:
df.loc[dificil, ["descripcion", "familia", "pred_regla", "pred_tfidf_palabras",
                 "pred_tfidf_caracteres", "motivo_dificultad"]]

,descripcion,familia,pred_regla,pred_tfidf_palabras,pred_tfidf_caracteres,motivo_dificultad
1,BRICK CARTON/PE/ALU 1L 28G,COMPLEJO_ALU,COMPLEJO_ALU,COMPLEJO_ALU,PAPEL_CARTON,Contiene 'CARTON' pero es envase complejo
5,BOLSA PE RECICLADO 30% PCR 60MIC 18G,PEBD,SIN_REGLA,PEBD,PEBD,Porcentaje de reciclado puede confundirse con ...
7,"TAPON CORONA ALUMINIO 26MM 1,9G",ALUMINIO,ALUMINIO,ACERO,ACERO,"Tapón corona normalmente acero, aquí aluminio"
14,ENVASE COMPUESTO CARTON+ALU TUBO 22G,COMPLEJO_ALU,COMPLEJO_ALU,PAPEL_CARTON,PAPEL_CARTON,Lata compuesta: cartón + aluminio
21,"CARTÖN ONDULADO CAJA 0,25 KG",PAPEL_CARTON,SIN_REGLA,PAPEL_CARTON,PAPEL_CARTON,Error de codificación (Ö) y peso en kg con esp...
27,"TAPON PP C/LINER ALU 2,3G",PP,COMPLEJO_ALU,COMPLEJO_ALU,ALUMINIO,Material mayoritario PP; el liner de aluminio ...
54,"FILM BOPP METALIZADO 20MIC 5,2G",PLASTICO_MULTICAPA,PLASTICO_MULTICAPA,PEBD,PEBD,"Metalizado: aluminio depositado al vacío, crit..."
67,"ETIQUETA ENVOLVENTE 1,5L",PP,SIN_REGLA,PP,PP,No menciona material ni peso
68,"PREFORMA 23,5G 28MM",PET,SIN_REGLA,PEAD,PET,"Sin material explícito; '28MM' es diámetro, no..."
74,"BOLSA ALU-FREE OPP/PE 6,4G",PLASTICO_MULTICAPA,COMPLEJO_ALU,PLASTICO_MULTICAPA,PLASTICO_MULTICAPA,Negación en inglés ('ALU-FREE')


## B4. De la predicción a la decisión: ¿qué se automatiza y qué se revisa?

El `score` no está calibrado y no demuestra que la predicción sea correcta (notebook 04). Comparamos dos controles para decidir qué propuesta pasa a revisión rápida y cuál a revisión completa:

1. **Umbral de score** del modelo de caracteres.
2. **Acuerdo regla–modelo:** solo se propone sin revisión detallada si dos métodos con lógicas distintas coinciden.

In [13]:
filas = []
for umbral in [0.3, 0.4, 0.5, 0.6]:
    auto = df["score_caracteres"] >= umbral
    filas.append({"Control": f"score ≥ {umbral}", "% propuesto sin revisión detallada": auto.mean(),
                  "Precisión en esos casos": (df.loc[auto, "pred_tfidf_caracteres"] == df.loc[auto, "familia"]).mean(),
                  "Casos difíciles que se escapan": int((auto & dificil).sum())})
acuerdo = df["pred_regla"] == df["pred_tfidf_caracteres"]
filas.append({"Control": "acuerdo regla–modelo", "% propuesto sin revisión detallada": acuerdo.mean(),
              "Precisión en esos casos": (df.loc[acuerdo, "pred_regla"] == df.loc[acuerdo, "familia"]).mean(),
              "Casos difíciles que se escapan": int((acuerdo & dificil).sum())})
pd.DataFrame(filas).set_index("Control").style.format({"% propuesto sin revisión detallada": "{:.1%}", "Precisión en esos casos": "{:.1%}"})

,% propuesto sin revisión detallada,Precisión en esos casos,Casos difíciles que se escapan
Control,,,
score ≥ 0.3,97.9%,94.1%,21
score ≥ 0.4,92.6%,94.9%,14
score ≥ 0.5,91.0%,96.5%,11
score ≥ 0.6,82.5%,98.1%,6
acuerdo regla–modelo,84.1%,99.4%,5


In [14]:
print("Errores que el control de acuerdo NO detecta:")
df.loc[acuerdo & (df["pred_regla"] != df["familia"]), ["descripcion", "familia", "pred_regla", "pred_tfidf_caracteres"]]

Errores que el control de acuerdo NO detecta:


,descripcion,familia,pred_regla,pred_tfidf_caracteres
175,"MEMBRANA SIN ALU PE/PET 1,1G",PLASTICO_MULTICAPA,COMPLEJO_ALU,COMPLEJO_ALU


**Interpretación.** El acuerdo regla–modelo reduce la revisión detallada a una minoría de componentes con muy pocos errores en el resto. El error que se escapa es el más instructivo: `MEMBRANA SIN ALU`. **Regla y modelo fallan igual** porque ambos ven `ALU` y ninguno representa la negación. Es el mismo fenómeno del notebook 01 (`no estuvo mal`) en un dominio industrial. Control adicional propuesto: una regla de alerta que mande a revisión cualquier descripción con `SIN`, `FREE`, `NO` o `LIBRE DE`.

# Parte C — Encontrar componentes ya validados parecidos

La propuesta de familia es más fácil de revisar si se acompaña de **evidencia**: los 3 componentes ya declarados que más se parecen. Es el buscador del notebook 03, con el histórico validado como corpus.

**Predicción:** para `LATA ALUMNIO 33CL` (con errata), ¿recuperará mejor latas un TF-IDF de palabras o uno de caracteres?

In [15]:
from sklearn.metrics.pairwise import cosine_similarity

vec_busqueda = vec_caracteres().fit(df["descripcion"])
matriz_corpus = vec_busqueda.transform(df["descripcion"])

def buscar_similares(consulta, top_k=3, excluir_id=None):
    scores = cosine_similarity(vec_busqueda.transform([consulta]), matriz_corpus)[0]
    resultado = df.assign(similitud=scores)
    if excluir_id is not None:
        resultado = resultado[resultado["id_componente"] != excluir_id]
    return resultado.nlargest(top_k, "similitud")[["id_componente", "descripcion", "familia", "similitud"]].round(3)

buscar_similares("LATA ALUMNIO 33CL 12,1G", excluir_id=df.loc[df["descripcion"] == "LATA ALUMNIO 33CL 12,1G", "id_componente"].iloc[0])

,id_componente,descripcion,familia,similitud
119,CMP-1119,LATA ALU 33CL,ALUMINIO,0.501
91,CMP-1091,"LATA ALUMINIO SLEEK 33CL 12,4G",ALUMINIO,0.446
184,CMP-1184,"LATA ALU 33CL 9,9G",ALUMINIO,0.382


In [16]:
# Excluimos el propio componente para simular que es nuevo
id_operculo = df.loc[df["descripcion"] == "OPERCULO PAPEL/PE 0,6G", "id_componente"].iloc[0]
buscar_similares("OPERCULO PAPEL/PE 0,6G", excluir_id=id_operculo)

,id_componente,descripcion,familia,similitud
31,CMP-1031,OPERCULO ALU/PP 99MM,COMPLEJO_ALU,0.425
139,CMP-1139,"OPERCULO ALU/PP 99MM 1,2G",COMPLEJO_ALU,0.382
151,CMP-1151,"OPERCULO ALU/PP 99MM 1,2GR",COMPLEJO_ALU,0.369


### Evaluación: ¿los vecinos comparten familia?

Con validación cruzada, cada descripción de prueba busca en el resto. Medimos:

- **hit@1:** el primer resultado es de la misma familia.
- **precision@3:** proporción de los 3 primeros que son de la misma familia.

In [17]:
def evaluar_busqueda(codificar):
    hit1, p3 = [], []
    familias = df["familia"].to_numpy(dtype=object)
    for tr, te in cv.split(df, df["familia"]):
        emb_tr, emb_te = codificar(df["descripcion"].iloc[tr].tolist(), df["descripcion"].iloc[te].tolist())
        top = np.argsort(-cosine_similarity(emb_te, emb_tr), axis=1)[:, :3]
        f_tr, f_te = familias[tr], familias[te]
        hit1 += list(f_tr[top[:, 0]] == f_te)
        p3 += list((f_tr[top] == f_te[:, None]).mean(axis=1))
    return {"hit@1": np.mean(hit1), "precision@3": np.mean(p3)}

def codificador_tfidf(fabrica):
    def codificar(textos_tr, textos_te):
        v = fabrica().fit(textos_tr)
        return v.transform(textos_tr), v.transform(textos_te)
    return codificar

resultados_busqueda = {
    "TF-IDF palabras": evaluar_busqueda(codificador_tfidf(vec_palabras)),
    "TF-IDF caracteres": evaluar_busqueda(codificador_tfidf(vec_caracteres)),
}
pd.DataFrame(resultados_busqueda).T.style.format("{:.1%}")

,hit@1,precision@3
TF-IDF palabras,90.5%,81.1%
TF-IDF caracteres,90.5%,82.9%


### Comparación con embeddings E5 (notebooks 02 y 03)

Reutilizamos el mismo modelo multilingüe del curso, con los prefijos `query:` y `passage:`. **Predicción:** E5 se entrenó con lenguaje natural, no con códigos como `BOPP/MOPP` o `PCO1881`. ¿Ganará en este dominio?

> Esta celda necesita descargar el modelo (Colab o entorno del curso con 00B ejecutado).

In [18]:
try:
    from sentence_transformers import SentenceTransformer
    modelo_e5 = SentenceTransformer(
        "intfloat/multilingual-e5-small", revision="fd1525a9fd15316a2d503bf26ab031a61d056e98",
        device="cpu", local_files_only=not EN_COLAB,
    )
    def codificar_e5(textos_tr, textos_te):
        emb_tr = modelo_e5.encode([f"passage: {t}" for t in textos_tr], normalize_embeddings=True, show_progress_bar=False)
        emb_te = modelo_e5.encode([f"query: {t}" for t in textos_te], normalize_embeddings=True, show_progress_bar=False)
        return emb_tr, emb_te
    resultados_busqueda["Embeddings E5"] = evaluar_busqueda(codificar_e5)
except Exception as error:
    print("E5 no disponible en este entorno:", type(error).__name__)

pd.DataFrame(resultados_busqueda).T.style.format("{:.1%}")

E5 no disponible en este entorno: ModuleNotFoundError


,hit@1,precision@3
TF-IDF palabras,90.5%,81.1%
TF-IDF caracteres,90.5%,82.9%


**Interpretación.** El TF-IDF de caracteres resiste erratas, abreviaturas y la mezcla de español, inglés y portugués porque comparte fragmentos como `ALU`, `CART` o `PEAD`. Si E5 no mejora, no significa que los embeddings sean inútiles: en este dominio la señal está en **códigos de material**, no en paráfrasis de lenguaje natural. Es el caso que el notebook 03 describía: "códigos y abreviaturas internas pueden requerir búsqueda léxica".

**Limitación.** Parecido no equivale a correcto: `OPERCULO PAPEL/PE` recupera opérculos con aluminio. Por eso la búsqueda **acompaña** a la propuesta y no la sustituye.

# Parte D — Salida integrada para el analista

Entrenamos el clasificador final con todo el histórico y combinamos las tres capacidades. Probamos con descripciones **nuevas**, que no están en el dataset.

In [19]:
modelo_final = clasificador(vec_caracteres()).fit(df["descripcion"], df["familia"])
PALABRAS_NEGACION = re.compile(r"\b(SIN|FREE|LIBRE|NO)\b|-FREE", re.IGNORECASE)

def proponer(descripcion):
    proba = modelo_final.predict_proba([descripcion])[0]
    familia_modelo = modelo_final.classes_[proba.argmax()]
    familia_regla = clasificar_regla(descripcion)
    peso, aviso_peso = extraer_peso(descripcion)
    similares = buscar_similares(descripcion, top_k=3)
    motivos = []
    if familia_regla != familia_modelo:
        motivos.append(f"regla dice {familia_regla}")
    if PALABRAS_NEGACION.search(descripcion):
        motivos.append("contiene negación")
    if aviso_peso:
        motivos.append(aviso_peso)
    return {
        "descripción": descripcion,
        "familia propuesta": familia_modelo,
        "score": round(float(proba.max()), 2),
        "peso extraído (g)": peso,
        "similares validados": " | ".join(f"{d} [{f}]" for d, f in zip(similares["descripcion"], similares["familia"])),
        "decisión": "REVISAR: " + "; ".join(motivos) if motivos else "Confirmación rápida",
    }

nuevos = [
    "BOTELLA PET 1L 30% RPET 24,5G",
    "TAPA LATA ALUMINIO 206 2,6G",
    "BOLSA SNACK OPP/MET-OPP 125G",
    "MEMBRANA ALU-FREE PET/PE 0,9G",
    "CAIXA CARTAO 6X1L 0,190KG",
    "TAPON 38MM AZUL 3,1G",
]
pd.set_option("display.max_colwidth", 120)
pd.DataFrame([proponer(t) for t in nuevos])

,descripción,familia propuesta,score,peso extraído (g),similares validados,decisión
0,"BOTELLA PET 1L 30% RPET 24,5G",PET,0.85,24.5,"BOTELLA PET 25% RPET 33CL [PET] | BOTELLA PET 25% RPET 1,5L 21G [PET] | BOTELLA PET 25% RPET 500ML [PET]",Confirmación rápida
1,"TAPA LATA ALUMINIO 206 2,6G",ALUMINIO,0.92,2.6,"LATA ALUMINIO 50CL 14,2G [ALUMINIO] | TAPA LATA ALU 202 [ALUMINIO] | LATA ALUMINIO SLEEK 33CL 9,6 g [ALUMINIO]",Confirmación rápida
2,BOLSA SNACK OPP/MET-OPP 125G,PLASTICO_MULTICAPA,0.94,125.0,BOLSA SNACK BOPP/BOPP MET [PLASTICO_MULTICAPA] | LAMINADO OPP/MET-OPP 150G PATATA [PLASTICO_MULTICAPA] | LAMINADO OP...,Confirmación rápida
3,"MEMBRANA ALU-FREE PET/PE 0,9G",PLASTICO_MULTICAPA,0.59,0.9,"BOLSA ALU-FREE OPP/PE 6,4G [PLASTICO_MULTICAPA] | MEMBRANA SIN ALU PE/PET 1,1G [PLASTICO_MULTICAPA] | MEMBRANA PET/A...",REVISAR: regla dice COMPLEJO_ALU; contiene negación
4,"CAIXA CARTAO 6X1L 0,190KG",PAPEL_CARTON,0.80,190.0,"CAIXA CARTAO CANELADO 317G [PAPEL_CARTON] | CAIXA CARTAO CANELADO 0,3032KG [PAPEL_CARTON] | CAIXA CARTAO CANELADO 19...",Confirmación rápida
5,"TAPON 38MM AZUL 3,1G",PEAD,0.38,3.1,"TAPON HDPE 38MM BLANCO 4G [PEAD] | TAPON HDPE 38MM BLANCO 3.5G [PEAD] | TAPON PEAD 28MM 1,8 g [PEAD]",REVISAR: regla dice SIN_REGLA


**Interpretación.** La salida no es una declaración, es una **ficha de revisión**: propuesta, evidencia (casos similares) y motivo para revisar. Dos lecturas críticas:

- `MEMBRANA ALU-FREE` acierta la familia porque el histórico ya contiene un ejemplo negado, pero se envía a revisión igualmente por la alerta de negación. El control no depende de que el modelo acierte.
- `BOLSA SNACK OPP/MET-OPP 125G` recibe "confirmación rápida" con un **peso falso**: 125 g es el contenido, no la bolsa. Es el error silencioso de la Parte A. Por eso "confirmación rápida" sigue siendo una confirmación humana, y la mejora prioritaria es contrastar el peso con el rango típico de la familia.

# Cierre

## ¿Qué ve ahora la máquina?

```text
DESCRIPCIÓN LIBRE → peso (regex) + familia (TF-IDF caracteres + LR, contrastada con regla)
                  + 3 casos validados similares → PROPUESTA → REVISIÓN HUMANA → DECLARACIÓN
```

Ve fragmentos de material, erratas y abreviaturas, y casos históricos parecidos. **No** ve la negación, el material cuando la descripción no lo dice, ni la diferencia entre peso del contenido y peso del envase.

## P6 — Fallo analizado: `MEMBRANA SIN ALU PE/PET`

| Pregunta | Respuesta |
|---|---|
| ¿Qué etapa o supuesto falló? | Representación: ni la regla ni TF-IDF modelan la negación; el supuesto "si aparece ALU, hay aluminio" es falso |
| ¿Qué error produjo? | Propone `COMPLEJO_ALU` para un multicapa plástico y, además, regla y modelo coinciden |
| ¿Qué coste puede tener? | Declaración en la familia equivocada (tarifa por kg distinta, error en la base imponible del plástico) |
| ¿Qué cambiaría primero? | El **proceso**: alerta por patrones de negación que fuerza revisión, además de añadir ejemplos negados al entrenamiento |
| ¿Cómo volvería a medir? | Conjunto de prueba con descripciones negadas y recall de las alertas sobre ellas |
| ¿Qué monitorizaría? | % de propuestas corregidas por el analista, por familia y por proveedor; vocabulario nuevo de proveedores |